#Phase 1: Loading the dataset, categorize it into 6 state and save into pickle

In [3]:
import pandas as pd
import numpy as np
import re
import pickle
from dataclasses import dataclass, field
from typing import List, Dict
from tqdm.auto import tqdm

In [4]:
@dataclass
class MiddleNode:
    summary: str
    anchors: set
    confidence: float
    embedding: np.ndarray
    scores: dict  # probabilistic branch scores


@dataclass
class Branch:
    name: str
    middle: MiddleNode
    evidence: List[str] = field(default_factory=list)


@dataclass
class Signature:
    schema_id: str
    stability: float
    coverage: float
    generation_allowed: bool
    author: str


@dataclass
class Case:
    case_id: int
    branches: Dict[str, Branch]
    signature: Signature


In [5]:
!git clone https://github.com/Bhuvanesh2218K/chaturya-vector-engine.git
%cd chaturya-vector-engine

Cloning into 'chaturya-vector-engine'...
remote: Enumerating objects: 162, done.
remote: Counting objects: 100% (19/19), done.
remote: Compressing objects: 100% (16/16), done.
remote: Total 162 (delta 4), reused 0 (delta 0), pack-reused 143 (from 1)
Receiving objects: 100% (162/162), 76.27 KiB | 5.45 MiB/s, done.
Resolving deltas: 100% (58/58), done.
/kaggle/working/chaturya-vector-engine


In [4]:
import sys
sys.path.append("/content/chaturya-vector-engine")

In [6]:
# embedding call

from chaturya_vector_engine.engine import ChaturyaVectorEngine
engine= ChaturyaVectorEngine(dim=512)

In [7]:
def embed_text(text: str):
    return engine.batch_encode([text])[0]

In [8]:
# Helpers
# split the sentence accoding to next line
def split_turns(conversation: str):
    lines=conversation.split("\n")
    return [l.strip() for l in lines if l.strip()]

def detect_speaker(line : str):
    line=line.lower()
    if "therapist" in line:
        return "therapist"
    else: return "patient"

In [9]:
def compute_branch_scores(text, state_a, emo_a, concerns_a, risk_a):
    total = max(len(state_a) + len(emo_a) + len(concerns_a) + len(risk_a), 1)

    scores = {
        "state": round(len(state_a)/total, 3),
        "emotion": round(len(emo_a)/total, 3),
        "concern": round(len(concerns_a)/total, 3),
        "risk": round(len(risk_a)/total, 3)
    }

    # distress intensity boost
    distress_words = {"panic","terrified","overwhelmed","hopeless","worthless"}
    if any(w in text for w in distress_words):
        scores["emotion"] += 0.10

    return scores


In [10]:
def build_clinical_summary(state_a, emo_a, concerns_a, risk_score):

    severity = (
        "severe distress"
        if risk_score >= 0.85 else
        "high distress"
        if risk_score >= 0.45 else
        "moderate distress"
    )

    state_part = (
        f"Patient reports {', '.join(sorted(state_a))}."
        if state_a else ""
    )

    emotion_part = (
        f"They describe feeling {', '.join(sorted(emo_a))}."
        if emo_a else ""
    )

    concern_part = (
        f"They express concerns about {', '.join(sorted(concerns_a))}."
        if concerns_a else ""
    )

    return " ".join([
        state_part,
        emotion_part,
        concern_part,
        f"Distress level appears {severity}."
    ]).strip()


In [11]:
# Anchor System (Hybrid: curated + dynamic)
SYMPTOMS = set([
    "sleep", "insomnia", "tired", "fatigue", "exhausted", "memory", "focus", 
    "headache", "pain", "lethargy", "brain fog", "migraine", "nausea", 
    "dizziness", "restless", "shaking", "appetite", "weight loss", "weight gain",
    "insomniac", "burnt out", "weakness", "drowsy", "low energy", "palpitations",
    "tension", "aching", "chest pressure", "breathless", "forgetful", "unfocused",
    "trembling", "nightmares", "sweating", "digestion", "numbness", "chills","breathing", "shortness of breath", "breathing difficulty",
"chest tight", "tight chest", "breathing problem"
])

EMOTIONS = set([
    "anxious", "sad", "stress", "fear", "overwhelmed", "angry", "lonely", 
    "hopeless", "depressed", "worthless", "guilty", "ashamed", "irritable", 
    "numb", "empty", "frustrated", "despair", "terrified", "panic", 
    "isolated", "rejected", "miserable", "heartbroken", "bitter", "pessimistic",
    "unmotivated", "agitated", "gloomy", "melancholy", "devastated", "broken",
    "apathetic", "helpless", "insecure", "paranoia", "doomscrolling", "resentful"
])

CONCERNS = set([
    "job", "work", "family", "future", "relationship", "money", "health", 
    "school", "finances", "career", "divorce", "breakup", "bills", "debt", 
    "housing", "rent", "grades", "exams", "parents", "children", "legal", 
    "insurance", "unemployment", "mortgage", "grief", "bereavement", 
    "lawsuit", "eviction", "bankruptcy", "bullying", "harassment", "trauma",
    "addiction", "alcohol", "drugs", "loneliness", "identity", "midlife crisis"
])

SELF_HARM_RISK = {
    "suicide", "kill myself", "end my life", "suicidal",
    "cutting", "overdose", "giving up", "better off dead",
    "death wish", "harming myself", "hurt myself",
    "no way out", "want to die", "can't go on",
    "nothing matters", "hanging", "bridge", "pills",
    "razor", "final goodbye", "not waking up"
}

MEDICAL_EMERGENCY = {
    "heart attack", "stroke", "seizure",
    "unconscious", "bleeding", "cannot breathe",
    "stabbing pain", "poison", "overdose"
}


STOPWORDS = set([
    "the", "is", "a", "an", "to", "and", "of", "in", "it", "that", "i", "am", 
    "was", "are", "this", "with", "from", "really", "very", "just", "basically", 
    "actually", "maybe", "sort of", "kind of", "me", "my", "so", "but", "if", 
    "or", "because", "as", "about", "your", "for", "been", "others", "support",
    "things", "people", "would", "could", "should", "going", "value", "human","provide", "seeking", "feeling", "others", "something", 
    "everything", "everyone", "someone", "pressure", "provide", 
    "getting", "making", "thought", "always", "really", "would",
    "could", "should", "things", "people"
])

In [12]:
CLINICAL_STATE_TERMS = {
    "fatigue","tired","exhausted","insomnia","sleep disturbance",
    "pain","dizziness","breathing difficulty","palpitations",
    "sweating","tingling","numbness","weakness","tension"
}

CLINICAL_CONCERN_TERMS = CONCERNS  # reuse your existing concern vocabulary

THERAPY_NOISE_WORDS = {
    "activities","become","between","navigate","sounds","important",
    "wonderful","process","journey","growth","experience","healing",
    "explore","sharing","understand","support","connection","personal",
    "things","stuff","something","anything","everything"
}

In [13]:
def dynamic_keywords(text: str, top_k=5):
    words = re.findall(r"[a-zA-Z]+", text.lower())

    NOISE_WORDS = {
        "therapy","journey","process","explore","experience","healing",
        "feeling","thinking","sharing","talking","working","trying",
        "things","something","anything","everything","someone",
        "really","maybe","yourself","through","related","sharing","situations",
        "solutions","remember","journey","explore","finding",
        "especially","activities","comparison","experiences",
        "things","stuff","something","anything",
        "navigate","important","sounds","become","between",
        "wonderful","process","growth","healing"
        }

    words = [
        w for w in words
        if w not in STOPWORDS
        and w not in NOISE_WORDS
        and len(w) > 5
        and w not in THERAPY_NOISE_WORDS
        and not w.endswith(("ly","ing","tion","ment"))
    ]

    freq = {}
    for w in words:
        freq[w] = freq.get(w, 0) + 1

    sorted_words = sorted(freq.items(), key=lambda x: -x[1])
    return [w for w, _ in sorted_words[:top_k]]


In [14]:
def extract_anchors(text: str, vocab):
    text_low = text.lower()

    curated = [w for w in vocab if w in text_low]
    dynamic = dynamic_keywords(text)

    anchors = list(set(curated + dynamic))

    GARBAGE = {
        "charlie","human","value","gpt",
        "therapy","journey","process","explore",
        "experience","healing","growth"
    }

    THERAPY_FILLER = {
    "myself","yourself","within","through","allowed","esteem",
    "journey","process","explore","experience","growth","healing",
    "relationships","connection","connections","personal",
    "feelings","thoughts","sharing","support","understand",
    "strategies","techniques","coping","tools","skills",
    "emotions","struggles","focus","balance","remember"
    }


    anchors = [
        w for w in anchors
        if w not in GARBAGE
        and w not in THERAPY_FILLER
        and len(w) > 2
    ]

    return set(anchors)


In [15]:
def build_middle(summary_text, anchors, embedding=None, scores=None):

    summary_text = summary_text.strip() or "no information"
    anchor_set = set(anchors)

    if len(anchor_set) == 0:
        confidence = 0.25
    elif len(anchor_set) <= 2:
        confidence = 0.55
    else:
        confidence = 0.80

    if embedding is None:
        embedding = embed_text(summary_text)

    if scores is None:
        scores = {}

    return MiddleNode(
        summary=summary_text,
        anchors=anchor_set,
        confidence=confidence,
        embedding=embedding,
        scores=scores
    )

In [16]:
# text cleaning helpers

def clean_text(text:str):
    words=re.findall(r"[a-zA-Z]+",text.lower())
    words=[w for w in words if w not in STOPWORDS]
    return " ".join(words)

Catergorizer --> Case Builder

In [17]:
def categorize_conversation(case_id: int, conversation: str):
    turns = split_turns(conversation)
    patient_lines, therapist_lines = [], []

    for t in turns:
        (patient_lines if detect_speaker(t) == "patient" else therapist_lines).append(t)

    patient_text = " ".join(patient_lines).lower()
    therapist_text = " ".join(therapist_lines).lower()

    # -------------------------------------------------
    # Anchor Extraction
    # -------------------------------------------------
    state_a   = extract_anchors(patient_text, SYMPTOMS)
    emo_a     = extract_anchors(patient_text, EMOTIONS)
    risk_a    = extract_anchors(patient_text, SELF_HARM_RISK)
    medical_a = extract_anchors(patient_text, MEDICAL_EMERGENCY)

    # -------------------------------------------------
    # Panic physiology & dissociation phrases
    # -------------------------------------------------
    panic_phrases = [
        "losing control",
        "feel unreal",
        "feel detached",
        "going crazy",
        "about to collapse",
        "heart racing"
    ]

    if any(p in patient_text for p in panic_phrases):
        emo_a.add("panic")

    # -------------------------------------------------
    # Risk Detection
    # -------------------------------------------------
    CRISIS_INTENT = {
        "kill myself", "end my life", "want to die",
        "suicide", "no reason to live", "can't go on"
    }

    PASSIVE_DISTRESS = {
        "drowning", "stop existing", "no point",
        "disappear", "can't handle anymore"
    }

    intent = any(p in patient_text for p in CRISIS_INTENT)
    passive = any(p in patient_text for p in PASSIVE_DISTRESS)

    risk_score = len(risk_a) * 0.15

    if intent:
        risk_score += 0.7
    elif passive:
        risk_score += 0.25

    if medical_a:
        risk_score += 0.25

    if "chest pain" in patient_text:
        risk_score += 0.25
    elif "chest tight" in patient_text or "tightness" in patient_text:
        risk_score += 0.1

    # distress intensity boost
    if len(emo_a) >= 3:
        risk_score += 0.05

    risk_score = min(1.0, risk_score)

    # -------------------------------------------------
    # Clinical Focus
    # -------------------------------------------------
    if risk_score >= 0.85:
        clinical_focus = "EMERGENCY_TRIAGE"
    elif risk_score >= 0.45:
        clinical_focus = "HIGH_DISTRESS_AFFECT"
    else:
        clinical_focus = "STABLE_CLINICAL_PRESENTATION"

    # -------------------------------------------------
    # Ensure state present
    # -------------------------------------------------
    if not state_a:
        if "breath" in patient_text or "breathing" in patient_text:
            state_a.add("breathing difficulty")
        elif "sleep" in patient_text:
            state_a.add("sleep disturbance")
        elif "tired" in patient_text or "exhausted" in patient_text:
            state_a.add("fatigue")

    # -------------------------------------------------
    # Primary Concerns
    # -------------------------------------------------
    primary_concerns = extract_anchors(patient_text, CONCERNS)

    if not primary_concerns:
        if "focus" in patient_text:
            primary_concerns.add("difficulty concentrating")
        if "fear" in patient_text or "scared" in patient_text:
            primary_concerns.add("fear response")
        if state_a:
            primary_concerns.update(state_a)

    # -------------------------------------------------
    # Embedding
    # -------------------------------------------------
    p_input = patient_text if len(patient_text) < 500 else patient_text[:250] + " " + patient_text[-250:]
    patient_embedding = embed_text(
        p_input + " " + " ".join(state_a) + " " + " ".join(emo_a)
    )

    # -------------------------------------------------
    # Probabilistic scoring
    # -------------------------------------------------
    branch_scores = compute_branch_scores(
        patient_text,
        state_a,
        emo_a,
        primary_concerns,
        risk_a
    )

    # -------------------------------------------------
    # Clinical conversational summary
    # -------------------------------------------------
    clinical_summary = build_clinical_summary(
        state_a,
        emo_a,
        primary_concerns,
        risk_score
    )

    # -------------------------------------------------
    # Therapist branch
    # -------------------------------------------------
    therapist_actions = extract_anchors(therapist_text, CONCERNS)

    # -------------------------------------------------
    # Build branches (store MiddleNode)
    # -------------------------------------------------
    branches = {}

    if state_a:
        branches["state"] = Branch(
            "state",
            build_middle(", ".join(state_a), state_a, patient_embedding),
            patient_lines
        )

    if emo_a:
        branches["emotion"] = Branch(
            "emotion",
            build_middle(", ".join(emo_a), emo_a, patient_embedding),
            patient_lines
        )

    if primary_concerns:
        branches["concern"] = Branch(
            "concern",
            build_middle(", ".join(primary_concerns), primary_concerns, patient_embedding),
            patient_lines
        )

    # risk anchors include medical signals
    risk_anchor_set = set(risk_a) | medical_a

    branches["risk"] = Branch(
        "risk",
        build_middle(
            clinical_summary,
            risk_anchor_set,
            patient_embedding,
            scores=branch_scores
        ),
        patient_lines
    )

    if therapist_actions:
        branches["therapist"] = Branch(
            "therapist",
            build_middle(", ".join(therapist_actions), therapist_actions, patient_embedding),
            therapist_lines
        )

    # -------------------------------------------------
    # Signature
    # -------------------------------------------------
    signature = Signature(
        schema_id="MEDGEMMA_PRO_V1",
        stability=float(1.0 - risk_score),
        coverage=len(branches) / 5,
        generation_allowed=(risk_score < 0.9),
        author="BHUVANESH2218K"
    )

    return Case(case_id, branches, signature)


In [18]:
import os, gc, pickle, json, pandas as pd
from tqdm import tqdm

# --- SETTINGS ---
INPUT_CSV = "/kaggle/input/synthetic-therapy-conversations-dataset/train.csv"
WORKING_PKL = "cases_stream.pkl"
META_FILE = "metadata.json"
BATCH_LIMIT = 15000  # Total rows to process in THIS run

# 1. LOAD METADATA (RESUME LOGIC)
start_idx = 0
if os.path.exists(META_FILE):
    with open(META_FILE, 'r') as f:
        meta = json.load(f)
        start_idx = meta.get("last_processed_row", 0)
    print(f"📍 Resuming from Row: {start_idx}")
else:
    print("🆕 No metadata found. Starting from Row 0.")

# 2. PROCESSING
print(f"🚀 Goal for this session: Row {start_idx + BATCH_LIMIT}")

# Use 'ab' (append) if file exists, 'wb' if starting fresh
mode = 'ab' if os.path.exists(WORKING_PKL) else 'wb'

with open(WORKING_PKL, mode) as f:
    # Use skiprows to skip exactly where we left off
    reader = pd.read_csv(INPUT_CSV, skiprows=range(1, start_idx + 1), nrows=BATCH_LIMIT, chunksize=3000)
    current_id = start_idx
    
    for chunk in reader:
        for conv in tqdm(chunk["conversations"], desc=f"Batch starting @ {current_id}"):
            # --- YOUR FUNCTION HERE ---
            case = categorize_conversation(current_id, conv) 
            # --------------------------
            pickle.dump(case, f)
            current_id += 1
        
        # Every 1000 rows, clear RAM
        gc.collect()

# 3. SAVE METADATA (FOR NEXT RESTART)
with open(META_FILE, "w") as f:
    json.dump({"last_processed_row": current_id}, f)

print(f"✅ Session Complete! Final Row: {current_id}")
print(f"💾 Check the sidebar: {WORKING_PKL} and {META_FILE} are ready.")

🆕 No metadata found. Starting from Row 0.
🚀 Goal for this session: Row 15000


Batch starting @ 12000: 100%|██████████| 3000/3000 [00:52<00:00, 57.34it/s]


✅ Session Complete! Final Row: 15000
💾 Check the sidebar: cases_stream.pkl and metadata.json are ready.


#Phase 2: Tree Builder

In [19]:
!pip install faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 71.3 MB/s eta 0:00:00:00:0100:01


In [20]:
import pickle
import numpy as np
import faiss
import os

cases_path = "/kaggle/working/chaturya-vector-engine/cases_stream.pkl"

# Aligned lists - these will all have the same length
middle_embeddings = []
node_refs = []
node_anchors = []
summary_cache = {} # (case_id, branch_name) -> summary

print("🔄 Processing cases and aligning forest...")

INDEXABLE_BRANCHES = {"state", "emotion", "risk", "concern"}

with open(cases_path, "rb") as f:
    while True:
        try:
            case = pickle.load(f)
            for name, branch in case.branches.items():

                # skip therapist & non-clinical branches
                if name not in INDEXABLE_BRANCHES:
                    continue

                if hasattr(branch.middle, 'embedding') and branch.middle.embedding is not None:

                    anchors = branch.middle.anchors if hasattr(branch.middle, 'anchors') else []
                    summary = getattr(branch.middle, "summary", "")

                    # filter low-signal nodes
                    if len(anchors) == 0 or len(summary.split()) < 4:
                       continue


                    middle_embeddings.append(np.asarray(branch.middle.embedding, dtype="float32"))
                    node_refs.append((case.case_id, name))
                    node_anchors.append(list(anchors))
                    summary_cache[(case.case_id, name)] = {
                        "summary": summary,
                        "anchors": anchors,
                        "scores": getattr(branch.middle, "scores", {})
                    }


        except EOFError:
            break


print(f"✅ Alignment Complete!")
print(f"Total Aligned Nodes: {len(middle_embeddings)}")

🔄 Processing cases and aligning forest...
✅ Alignment Complete!
Total Aligned Nodes: 53789


#PHASE 3 — Retrieval + MedGemma Pipeline

In [21]:
# 1. Convert to Matrix
emb_matrix = np.vstack(middle_embeddings).astype("float32")
print("Embedding matrix shape:", emb_matrix.shape)

# 2. Normalize for Cosine Similarity
faiss.normalize_L2(emb_matrix)

# 3. Build FAISS Index
d = emb_matrix.shape[1]
index = faiss.IndexFlatIP(d)
index.add(emb_matrix)

# 4. Save Index
faiss.write_index(index, "faiss.index")
print(f"🚀 FAISS index built and saved for {index.ntotal} vectors.")

Embedding matrix shape: (53789, 512)
🚀 FAISS index built and saved for 53789 vectors.


In [22]:
# Save the node references
with open("node_refs.pkl", "wb") as f:
    pickle.dump(node_refs, f)

# Save the summary cache
with open("summary_cache.pkl", "wb") as f:
    pickle.dump(summary_cache, f)

# Save the anchors
with open("node_anchors.pkl", "wb") as f:
    pickle.dump(node_anchors, f)

print("💾 All aligned metadata files saved:")
print(f"- node_refs.pkl ({len(node_refs)} items)")
print(f"- summary_cache.pkl ({len(summary_cache)} items)")
print(f"- node_anchors.pkl ({len(node_anchors)} items)")

# FINAL ALIGNMENT CHECK
if index.ntotal == len(node_refs) == len(node_anchors):
    print("\n🌟 SYSTEM FULLY SYNCED. Ready for RAG.")
else:
    print("\n⚠️ ALIGNMENT ERROR: Counts still do not match. Check the loops.")

💾 All aligned metadata files saved:
- node_refs.pkl (53789 items)
- summary_cache.pkl (53789 items)
- node_anchors.pkl (53789 items)

🌟 SYSTEM FULLY SYNCED. Ready for RAG.


In [23]:
!kaggle datasets init -p rag_assets

Traceback (most recent call last):
  File "/usr/local/bin/kaggle", line 4, in <module>
    from kaggle.cli import main
  File "/usr/local/lib/python3.12/dist-packages/kaggle/__init__.py", line 6, in <module>
    api.authenticate()
  File "/usr/local/lib/python3.12/dist-packages/kaggle/api/kaggle_api_extended.py", line 434, in authenticate
    raise IOError('Could not find {}. Make sure it\'s located in'
OSError: Could not find kaggle.json. Make sure it's located in /root/.config/kaggle. Or use the environment method. See setup instructions at https://github.com/Kaggle/kaggle-api/


In [22]:
def anchor_filter_nodes(user_text, max_nodes=200):
    """
    Pre-filter nodes using anchor overlap for speed & relevance.
    Uses partial matching to improve recall.
    """

    user_anchor = set(dynamic_keywords(user_text, top_k=8))
    matched_ids = []

    for i, anchors in enumerate(node_anchors):

        # improved matching (handles variations & substrings)
        if any(a in anc or anc in a for a in user_anchor for anc in anchors):
            matched_ids.append(i)

        if len(matched_ids) >= max_nodes:
            break

    return matched_ids, user_anchor


In [23]:
def faiss_rank(user_text, top_k=5):
    vec = embed_text(user_text).astype("float32").reshape(1, -1)
    faiss.normalize_L2(vec)
    _, I = index.search(vec, top_k)
    return I[0]


In [24]:
def faiss_rank_filtered(user_text, candidate_ids, top_k=5):
    """
    Rank only candidate nodes using cosine similarity.
    Falls back to full FAISS search if no candidates.
    """

    if not candidate_ids:
        return faiss_rank(user_text, top_k)

    # if candidates are very few, skip ranking overhead
    if len(candidate_ids) <= top_k:
        return candidate_ids

    user_vec = embed_text(user_text).astype("float32").reshape(1, -1)
    faiss.normalize_L2(user_vec)

    candidate_vecs = emb_matrix[candidate_ids]

    scores = np.dot(candidate_vecs, user_vec.T).flatten()
    top_indices = np.argsort(scores)[::-1][:top_k]

    return [candidate_ids[i] for i in top_indices]


In [25]:
def load_cases_by_ids(target_case_ids):
    """
    Stream load only required cases (memory efficient).
    """

    target_case_ids = set(target_case_ids)
    selected = {}

    with open(cases_path, "rb") as f:
        while True:
            try:
                case = pickle.load(f)

                if case.case_id in target_case_ids:
                    selected[case.case_id] = case

                if len(selected) == len(target_case_ids):
                    break

            except EOFError:
                break

    return selected


In [26]:
def build_context_from_nodes(node_ids, max_cases=3):
    """
    Build RAG context using summaries + probabilistic scores.
    Prevents duplicate case usage.
    """

    context = []
    used_cases = set()

    for idx in node_ids:
        cid, branch = node_refs[idx]

        if cid in used_cases:
            continue

        data = summary_cache.get((cid, branch))
        if not data:
            continue

        summary = data.get("summary", "")
        scores = data.get("scores", {})

        if not summary or summary == "no information":
            continue

        score_line = f" | scores={scores}" if scores else ""

        context.append(f"{branch.upper()}: {summary}{score_line}")
        used_cases.add(cid)

        if len(used_cases) >= max_cases:
            break

    return "\n".join(context)


In [27]:
def retrieve_context(user_query, top_k=10):
    """
    Full retrieval pipeline:
    1. anchor pre-filter
    2. semantic ranking
    3. context building
    """

    candidate_ids, anchors = anchor_filter_nodes(user_query)

    print("Anchor keywords:", anchors)
    print("Candidate nodes after anchor:", len(candidate_ids))

    node_ids = faiss_rank_filtered(user_query, candidate_ids, top_k)
    print("Top nodes:", node_ids)

    context = build_context_from_nodes(node_ids)

    # 🔥 Always include user query (prevents empty context drift)
    context = f"USER_INPUT: {user_query}\n\n" + context

    print("\n===== RAW RETRIEVAL SIGNALS =====\n")
    print(context)

    return context


In [28]:
query="my body feels wrong and unreal. my chest is tight, breathing feels strange, and my heart is racing. i feel like i’m losing control and might disappear. i’m scared something is seriously wrong and i can’t calm myself down."
context = retrieve_context(query, top_k=10)

print("\n===== CONTEXT TO MEDGEMMA =====\n")
print(context[:2000])

Anchor keywords: {'myself', 'disappear', 'scared', 'strange', 'control', 'unreal'}
Candidate nodes after anchor: 200
Top nodes: [609, 608, 606, 607, 2844, 2846, 2845, 3305, 3303, 3304]

===== RAW RETRIEVAL SIGNALS =====

USER_INPUT: my body feels wrong and unreal. my chest is tight, breathing feels strange, and my heart is racing. i feel like i’m losing control and might disappear. i’m scared something is seriously wrong and i can’t calm myself down.

RISK: Patient reports aching, constant, control, exhausted. They describe feeling ashamed, constant, control, stress. They express concerns about constant, control, health, relationship, work. Distress level appears high distress. | scores={'state': 0.267, 'emotion': 0.267, 'concern': 0.333, 'risk': 0.133}
STATE: anxiety, aching, shortness of breath, control, breathing
RISK: Patient reports afraid, behind, handle, scared, weakness. They describe feeling afraid, behind, fear, handle, scared. They express concerns about afraid, behind, hand

Clinical Interpreter

In [29]:
import re

def normalize_text(text: str) -> str:
    """Remove conversational noise"""
    text = text.lower()
    text = re.sub(r"from human value", "", text)
    text = re.sub(r"from gpt value", "", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

In [30]:

PANIC_LANGUAGE = {
    "losing control", "lose control", "freaking out",
    "out of control", "can't calm down", "cant calm down"
}

DISSOCIATION_LANGUAGE = {
    "disappear", "not real", "feel unreal",
    "detached", "floating", "out of my body"
}

CATASTROPHIC_FEAR = {
    "something is wrong",
    "something terrible",
    "something bad is happening"
}

OVERWHELM_LANGUAGE = {
    "too much",
    "can't handle", "cant handle",
    "overwhelming",
    "everything at once"
}



def semantic_anchor_match(text: str):
    """
    Detect psychological distress cognition patterns.
    Designed for therapy conversation context.
    """

    text = text.lower()
    detected = {"emotion": set()}

    if any(p in text for p in PANIC_LANGUAGE):
        detected["emotion"].update({"panic", "anxiety"})

    if any(p in text for p in DISSOCIATION_LANGUAGE):
        detected["emotion"].update({"panic", "fear"})

    if any(p in text for p in CATASTROPHIC_FEAR):
        detected["emotion"].add("fear")

    if any(p in text for p in OVERWHELM_LANGUAGE):
        detected["emotion"].add("overwhelmed")

    return detected


In [31]:
def build_clean_context(node_ids, node_refs, summary_cache, current_query=""):

    extracted = {
        "state": set(),
        "emotion": set(),
        "concern": set(),
        "risk": set()
    }

    retrieval_distress_flag = False

    # =====================================================
    # 1️⃣ USER QUERY SIGNALS (PRIMARY)
    # =====================================================
    if current_query:
        q = current_query.lower()

        # -------- physical sensations (panic physiology) --------
        if "breath" in q or "breathing" in q or "short of breath" in q:
            extracted["state"].add("breathing difficulty")

        if "chest tight" in q or "tightness" in q:
            extracted["state"].add("chest tightness")

        if "heart racing" in q or "heart pounding" in q:
            extracted["state"].add("palpitations")

        if "sweaty" in q or "sweating" in q:
            extracted["state"].add("sweating")

        if "dizzy" in q or "lightheaded" in q:
            extracted["state"].add("dizziness")

        if "tingly" in q or "numb" in q:
            extracted["state"].add("tingling sensation")

        # -------- emotion detection --------
        for e in EMOTIONS:
            if e in q:
                extracted["emotion"].add(e)

        if "fear" in q or "scared" in q:
            extracted["emotion"].add("fear")

        if "anxiety" in q or "anxious" in q:
            extracted["emotion"].add("anxiety")

        # semantic distress cognition
        semantic = semantic_anchor_match(current_query)
        extracted["emotion"].update(semantic["emotion"])

        # -------- cognitive distress --------
        if "can't think" in q or "cant think" in q:
            extracted["concern"].add("difficulty concentrating")

        if "something is wrong" in q:
            extracted["concern"].add("health worry")

        if "losing control" in q:
            extracted["concern"].add("fear of losing control")

        # -------- life concerns --------
        for c in CONCERNS:
            if c in q:
                extracted["concern"].add(c)

        if "family" in q or "kids" in q:
            extracted["concern"].add("family responsibilities")

        # -------- safety risk --------
        extracted["risk"].update([r for r in SELF_HARM_RISK if r in q])

    # =====================================================
    # 2️⃣ RETRIEVAL ENRICHMENT (THERAPY CONTEXT)
    # =====================================================
    for idx in node_ids:
        case_id, branch_type = node_refs[idx]
        meta = summary_cache.get((case_id, branch_type))

        if not meta:
            continue

        if branch_type == "emotion":
            retrieval_distress_flag = True

        elif branch_type == "concern":
            extracted["concern"].update(meta.get("anchors", []))

        elif branch_type == "state":
            if meta.get("anchors"):
                extracted["state"].update(meta["anchors"])

    # =====================================================
    # 3️⃣ CLINICAL INTERPRETATION (THERAPY SAFE)
    # =====================================================
    panic_cluster = {
        "breathing difficulty",
        "chest tightness",
        "palpitations",
        "sweating",
        "dizziness",
        "tingling sensation"
    }

    if len(extracted["state"].intersection(panic_cluster)) >= 2:
        extracted["concern"].add("panic-like physiological response")

    if retrieval_distress_flag and not extracted["emotion"]:
        extracted["emotion"].add("emotional distress")

    # =====================================================
    # 4️⃣ CLEANUP
    # =====================================================
    NOISE = {
        "journey","process","explore","experience","growth",
        "healing","sharing","things","something","anything"
    }

    for key in extracted:
        extracted[key] = {
            w for w in extracted[key]
            if w and w not in NOISE and len(w) > 2
        }

    # =====================================================
    # 5️⃣ FORMAT OUTPUT
    # =====================================================
    def format_line(items):
        return "- " + ", ".join(sorted(items)) if items else "- None detected"

    return f"""CLINICAL CONTEXT SUMMARY
-----------------------
Patient State:
{format_line(extracted['state'])}

Emotional Signals:
{format_line(extracted['emotion'])}

Primary Concerns:
{format_line(extracted['concern'])}

Risk Assessment:
{format_line(extracted['risk'])}""".strip()


In [32]:
query="my body feels wrong and unreal. my chest is tight, breathing feels strange, and my heart is racing. i feel like i’m losing control and might disappear. i’m scared something is seriously wrong and i can’t calm myself down."

In [33]:
# ==========================================
# FINAL SELF-CONTAINED SAFETY TEST
# ==========================================

# 1. Define the High-Risk Query
# 2. Get Node IDs from FAISS (The Search Step)
# This was the missing link causing the NameError
node_ids = faiss_rank(query, top_k=10) 

# 3. Build the Context with the CURRENT_QUERY safety check
# This ensures "end it all" is caught even if history is clean
context = build_clean_context(
    node_ids=node_ids, 
    node_refs=node_refs, 
    summary_cache=summary_cache, 
    current_query=query  # 🔥 CRITICAL: Pass the query here!
)

print("-" * 30)
print("FINAL CONTEXT SENT TO MEDGEMMA:")
print("-" * 30)
print(context)

------------------------------
FINAL CONTEXT SENT TO MEDGEMMA:
------------------------------
CLINICAL CONTEXT SUMMARY
-----------------------
Patient State:
- aching, breathing difficulty, childhood, exhausted, insecurities, noticed, pain, partner, relationship, specific, substances, tension

Emotional Signals:
- anxiety, fear, panic

Primary Concerns:
- addiction, alcohol, childhood, drugs, excited, fear of losing control, future, health, insecurities, job, loneliness, noticed, parents, partner, relationship, rent, specific, substances, together, work

Risk Assessment:
- None detected


In [ ]:
!pip install -q --upgrade \
    transformers \
    accelerate \
    bitsandbytes \
    sentencepiece


In [ ]:
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)

print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))


In [ ]:
MODEL_ID = "unsloth/medgemma-4b-it-bnb-4bit"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    use_fast=True
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)

model.eval()

print("✅ MedGemma loaded successfully")


In [ ]:
SYSTEM_PROMPT = """
You are MedGemma, a clinical support AI.

You are NOT a replacement for a clinician.

Your role:
• provide calm, empathetic support
• explain symptoms clearly
• reduce fear and uncertainty
• encourage professional care when appropriate

Guidelines:
• prioritize safety
• avoid medical diagnosis certainty
• do NOT hallucinate conditions
• ask gentle clarifying questions when helpful
"""

In [ ]:
@torch.inference_mode()
def generate_medgemma_response(user_query: str, clinical_context: str):

    grounded_context = f"""
PATIENT HISTORY & CLINICAL PATTERN SUMMARY

This information represents patterns observed in similar patients
and should be used to understand the current patient's condition.

Use this background to guide your response and maintain continuity.

IMPORTANT:
• Treat this as patient history, not optional reference.
• Use it to interpret symptoms and emotional patterns.
• Align your response with this context.

{clinical_context}
"""

    prompt = f"""
{SYSTEM_PROMPT}

{grounded_context}

--------------------------------
CURRENT PATIENT MESSAGE
--------------------------------
{user_query}

--------------------------------
RESPONSE INSTRUCTIONS
--------------------------------
• Be calm, supportive, and clear.
• Maintain continuity with the patient's prior messages.
• Acknowledge physical sensations AND emotions.
• If symptoms may relate to anxiety/panic, explain gently.
• If symptoms could require medical attention, suggest evaluation calmly.
• Encourage grounding or breathing if distress is high.
• Do NOT sound alarmist.
• Do NOT give definitive diagnosis.

RESPONSE:
"""

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    output = model.generate(
        **inputs,
        max_new_tokens=220,
        temperature=0.35,
        top_p=0.9,
        repetition_penalty=1.15,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.eos_token_id
    )

    text = tokenizer.decode(output[0], skip_special_tokens=True)

    if "RESPONSE:" in text:
        response = text.split("RESPONSE:")[-1].strip()
    else:
        response = text.strip()

    for marker in [
        "CURRENT PATIENT MESSAGE",
        "RESPONSE INSTRUCTIONS",
        "PATIENT HISTORY & CLINICAL PATTERN SUMMARY"
    ]:
        if marker in response:
            response = response.split(marker)[0].strip()

    if not response.endswith(('.', '!', '?')):
        last = max(response.rfind('.'), response.rfind('!'), response.rfind('?'))
        if last > 0:
            response = response[:last+1]

    return response

In [ ]:
CRISIS_RESOURCES = """
--------------------------------------------------
🆘 **IMMEDIATE SUPPORT AVAILABLE**
If you are feeling overwhelmed or thinking about self-harm, please reach out for help:
* **Call or Text:** 988 (Suicide & Crisis Lifeline - USA)
* **Text:** HOME to 741741 (Crisis Text Line)
* **Web:** 988lifeline.org
* *You are not alone. Support is available 24/7.*
--------------------------------------------------
"""

In [ ]:
def detect_crisis_risk(text: str) -> bool:
    crisis_terms = [
        "suicide", "kill myself", "want to die",
        "end my life", "no reason to live",
        "better off dead", "can't go on"
    ]
    text = text.lower()
    return any(term in text for term in crisis_terms)

In [ ]:
def answer_chatbot_query(user_query: str):

    # 1️⃣ Anchor filter + FAISS ranking
    candidate_ids, _ = anchor_filter_nodes(user_query)
    node_ids = faiss_rank_filtered(user_query, candidate_ids, top_k=5)

    # 2️⃣ Build clinical context
    clinical_context = build_clean_context(
        node_ids=node_ids,
        node_refs=node_refs,
        summary_cache=summary_cache,
        current_query=user_query
    )

    # 3️⃣ Generate grounded response
    response = generate_medgemma_response(
        user_query,
        clinical_context
    )

    # 4️⃣ Crisis safety layer
    if detect_crisis_risk(user_query + " " + clinical_context):
        response += "\n\n" + CRISIS_RESOURCES

    return response

In [ ]:
def print_session_history():
    print("\n🧠 SESSION MEMORY")
    print("-" * 40)

    if not chat_history:
        print("No history stored.")
        return

    for i, turn in enumerate(chat_history, 1):
        print(f"\nTurn {i}")
        print(f"User      : {turn['user']}")
        print(f"MedGemma  : {turn['assistant']}")

In [ ]:
def start_clinical_session():
    global chat_history
    chat_history = [] # Reset history for a new patient
    
    print("🏥 MedGemma Clinical Session Started (Type 'exit' to end)")
    print("-" * 50)
    
    while True:
        user_input = input("User: ")
        
        if user_input.lower() in ['exit', 'quit', 'bye']:
            print("MedGemma: Take care. Closing session.")
            break
            
        # Get response using our stateful function
        response = answer_chatbot_query(user_input)
        response = response.split("```")[0].strip()
        print(f"\nMedGemma: {response}")
        print("-" * 50)
        print_session_history()
# Start the chat
start_clinical_session()

In [23]:
def print_case_tree(case, show_evidence=False):
    print(f"\n🧠 Clinical Case Tree → ID: {case.case_id}")
    print("─" * 45)

    for branch_name, branch in case.branches.items():
        print(f"\n🔹 {branch_name.upper()}")

        middle = branch.middle

        # summary
        if hasattr(middle, "summary"):
            print(f"   Summary : {middle.summary}")

        # anchors
        if hasattr(middle, "anchors"):
            anchors = list(middle.anchors)[:8]
            print(f"   Anchors : {anchors}")

        # confidence
        if hasattr(middle, "confidence"):
            print(f"   Confidence : {middle.confidence}")

        # scores (for risk branch)
        if hasattr(middle, "scores") and middle.scores:
            print(f"   Scores : {middle.scores}")

        if show_evidence and branch.evidence:
            print("   Evidence:")
            for line in branch.evidence[:2]:
                print(f"     • {line}")

In [24]:
# load first case for demo
with open(cases_path, "rb") as f:
    case = pickle.load(f)

print_case_tree(case)


🧠 Clinical Case Tree → ID: 0
─────────────────────────────────────────────

🔹 STATE
   Summary : stress, challenges, supervisor, weakness, workload
   Anchors : ['stress', 'challenges', 'supervisor', 'weakness', 'workload']
   Confidence : 0.8

🔹 EMOTION
   Summary : stress, overwhelmed, challenges, supervisor, sad, workload
   Anchors : ['stress', 'overwhelmed', 'challenges', 'supervisor', 'sad', 'workload']
   Confidence : 0.8

🔹 CONCERN
   Summary : stress, work, challenges, supervisor, health, workload
   Anchors : ['stress', 'work', 'challenges', 'supervisor', 'health', 'workload']
   Confidence : 0.8

🔹 RISK
   Summary : Patient reports challenges, stress, supervisor, weakness, workload. They describe feeling challenges, overwhelmed, sad, stress, supervisor, workload. They express concerns about challenges, health, stress, supervisor, work, workload. Distress level appears severe distress.
   Anchors : ['supervisor', 'stress', 'challenges', 'workload']
   Confidence : 0.8
   Sco

In [25]:
def preview_case_forest(cases_path, n=3):
    print("\n🌲 Clinical Memory Forest Preview")
    print("=" * 50)

    with open(cases_path, "rb") as f:
        for _ in range(n):
            try:
                case = pickle.load(f)
                print_case_tree(case)
            except EOFError:
                break

In [26]:
preview_case_forest(cases_path, n=2)


🌲 Clinical Memory Forest Preview

🧠 Clinical Case Tree → ID: 0
─────────────────────────────────────────────

🔹 STATE
   Summary : stress, challenges, supervisor, weakness, workload
   Anchors : ['stress', 'challenges', 'supervisor', 'weakness', 'workload']
   Confidence : 0.8

🔹 EMOTION
   Summary : stress, overwhelmed, challenges, supervisor, sad, workload
   Anchors : ['stress', 'overwhelmed', 'challenges', 'supervisor', 'sad', 'workload']
   Confidence : 0.8

🔹 CONCERN
   Summary : stress, work, challenges, supervisor, health, workload
   Anchors : ['stress', 'work', 'challenges', 'supervisor', 'health', 'workload']
   Confidence : 0.8

🔹 RISK
   Summary : Patient reports challenges, stress, supervisor, weakness, workload. They describe feeling challenges, overwhelmed, sad, stress, supervisor, workload. They express concerns about challenges, health, stress, supervisor, work, workload. Distress level appears severe distress.
   Anchors : ['supervisor', 'stress', 'challenges', 'wor

In [27]:
from collections import Counter

def print_index_structure(node_refs):
    print("\n🌳 Retrieval Tree Structure")
    print("=" * 40)

    branch_counts = Counter(branch for _, branch in node_refs)

    for branch, count in branch_counts.items():
        print(f"{branch.upper():<10} → {count} nodes")

    print("\nTotal Indexed Nodes:", len(node_refs))

In [28]:
print_index_structure(node_refs)


🌳 Retrieval Tree Structure
STATE      → 11554 nodes
EMOTION    → 12825 nodes
CONCERN    → 14429 nodes
RISK       → 14981 nodes

Total Indexed Nodes: 53789


In [30]:
def print_visual_tree(case):
    print(f"\nClinical Case {case.case_id}")
    print("│")
    for name in case.branches:
        print(f"├── {name}")

In [31]:
print_visual_tree(case)


Clinical Case 0
│
├── state
├── emotion
├── concern
├── risk
